# HT-NN model for Sandbox

- Each sandbox contains 8 injection tubes and 8 injection tubes
- Input dimension for HT-NN: 8x8 (inj_events * obs_numb)
- Output dimension for HT-NN: 44x26x17 (core element size for sandbox )

In [4]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from utilies import *
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR, MultiStepLR, CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch is is_available on "{device}"')

torch is is_available on "cuda"


## HT-NN model

In [1]:
# Conv2d/Pooling calculator for rectangular
intput_size = (6, 6)
kernel_size = (3, 2)
stride = (1, 2)
padding = (0, 0)
H = 1 + (intput_size[0] + 2*padding[0]-(kernel_size[0]-1)-1)/stride[0]
W = 1 + (intput_size[1] + 2*padding[1]-(kernel_size[1]-1)-1)/stride[1]

print(f'output size H:{int(H)},  W:{int(W)}')

output size H:4,  W:3


In [2]:
# Convt2d calculator for rectangular
input_size = (7, 11)
kernel_size = (3, 2)
stride = (2, 2)
padding = (1, 0)
output_padding = (0, 0)  # optional

H = (input_size[0] - 1) * stride[0] - 2 * padding[0] + kernel_size[0] + output_padding[0]
W = (input_size[1] - 1) * stride[1] - 2 * padding[1] + kernel_size[1] + output_padding[1]

print(f'output size H:{int(H)},  W:{int(W)}')

output size H:13,  W:22


In [5]:
class SB_Net(nn.Module):
    """
    Atttributes
    -----------
        in_channels = 1 -> use only steady time (dh)
        init_features = 36 -> first conv output features
        out_channels = 18 -> 18 layers
        growth_rate  = 16 -> addtional features added in each densely connection
        layer_num = [3,3,...] -> number of layers in each dense block
        drop_rate = 0. -> dropout rate in each layer
    
    Input:
    -------
        x: tensor: (batch_size, 1, 8, 8)
    Output:
    -----------
        y: tensor: (batch_size, 17, 26, 44) # z , y, x
    """
    def __init__(self, 
                 in_channels,
                 init_features,
                 out_channels,
                 growth_rate,
                 layer_num,
                 drop_rate = 0.):
        super(SB_Net, self).__init__()

        self.out_channels = out_channels
        self.in_channels = in_channels
        self.init_features = init_features

        # (8, 8)
        self.in_conv = nn.Conv2d(in_channels, init_features, kernel_size=3, stride=1, padding=1, bias=False)
        num_features = init_features

        # (8, 8) => (7, 7)
        self.DTE1 = Dense_Transition_Encoder_block(init_features, growth_rate, layer_num[0], (2, 1, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[0] * growth_rate) // 2

        # (7, 7) => (6, 6)
        self.DTE2 = Dense_Transition_Encoder_block(num_features, growth_rate, layer_num[1], (2, 1, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[1] * growth_rate) // 2

        # (6, 6) => (5, 5)
        self.DTE3 = Dense_Transition_Encoder_block(num_features, growth_rate, layer_num[2], (2, 1, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[2] * growth_rate) // 2

        # (5, 5) => (4, 4)
        self.DTE4 = Dense_Transition_Encoder_block(num_features, growth_rate, layer_num[3], (2, 1, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[2] * growth_rate) // 2

# switch to decoder

        # (4, 4) => (5, 5)
        self.DTD1 = Dense_Transition_Decoder_block(num_features, growth_rate, layer_num[4], (2, 1, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[4] * growth_rate) // 2

        # (5, 5) => (7, 11)
        self.DTD2 = Dense_Transition_Decoder_block(num_features, growth_rate, layer_num[5], ((3, 3), (1, 2), (0, 0)), drop_rate=drop_rate)
        num_features = (num_features + layer_num[5] * growth_rate) // 2

        # (7, 11) => (13, 22)
        self.DTD3 = Dense_Transition_Decoder_block(num_features, growth_rate, layer_num[6], ((3, 2), (2, 2), (1, 0)), drop_rate=drop_rate)
        num_features = (num_features + layer_num[6] * growth_rate) // 2

        # (13, 22) => (26, 44)
        self.DTD4 = Dense_Transition_Decoder_block(num_features, growth_rate, layer_num[7], (2, 2, 0), drop_rate=drop_rate)
        num_features = (num_features + layer_num[7] * growth_rate) // 2

        self.out_conv = nn.Conv2d(num_features, self.out_channels, kernel_size=3, stride=1, padding=1, bias=False)        
        self.out_activation = Activation_block(negative_slope=0.5)

    def forward(self, x):

        x = self.in_conv(x)
        # print(f'after "channel_expand_conv2d": {x.shape}')
        x = self.DTE1(x)
        # print(f'after "DTE1": {x.shape}')
        x = self.DTE2(x)
        # print(f'after "DTE2": {x.shape}')
        x = self.DTE3(x)
        # print(f'after "DTE3": {x.shape}')
        x = self.DTE4(x)
        # print(f'after "DTE4": {x.shape}')

        x = self.DTD1(x)
        # print(f'after "DTD1": {x.shape}')
        x = self.DTD2(x)
        # print(f'after "DTD2": {x.shape}')
        x = self.DTD3(x)
        # print(f'after "DTD3": {x.shape}')
        x = self.DTD4(x)
        # print(f'after "DTD4": {x.shape}')      
        x = self.out_activation(x)
        y = self.out_conv(x)

        return y     

In [6]:
# check 
# Instantiate test
model_test = SB_Net(
            in_channels=1,     # Adjust based on your input channels
            init_features=54,
            out_channels=17,
            growth_rate=12,
            layer_num= [2]*8, # Adjust based on your architecture
            # base_channels= 64,
            drop_rate=0.)

dummy_input = torch.randn(1, 1, 8, 8) # Random test data

# Run through the model
output = model_test(dummy_input)
print(output.shape)

torch.Size([1, 17, 26, 44])
